In [2]:
from pathlib import Path

md_path = Path("/data/chengxi/Haystack/Command-Reference.md")
md_text = md_path.read_text(encoding="utf-8")



In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
chunk_sonic_command_reference.py

Split SONiC Command-Reference.md into CLI-semantic chunks suitable for Haystack ingestion.

Input : Command-Reference.md
Output: chunks.jsonl  (JSON Lines; one chunk per line)

Design goals:
- Prefer semantic boundaries:
  1) Level-2 headings: "## ..."
  2) Bold command markers: "**show xxx**", "**config yyy**" (common in SONiC docs)
  3) Fenced code blocks (``` ... ```) are preserved inside chunks
- Produce stable IDs (sha1 of normalized content + section title)
"""

import re
import json
import hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict


@dataclass
class Chunk:
    id: str
    title: str
    section: str
    command: Optional[str]
    content: str
    meta: Dict


# -----------------------------
# Helpers
# -----------------------------
H2_RE = re.compile(r"^##\s+(?P<title>.+?)\s*$", re.M)
BOLD_CMD_RE = re.compile(r"^\*\*(?P<cmd>[^*]+?)\*\*\s*$", re.M)  # lines like **show macsec**
FENCE_RE = re.compile(r"^```.*?$", re.M)


def normalize_for_id(text: str) -> str:
    # Stable but conservative normalization for hashing
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text


def sha1_id(*parts: str) -> str:
    h = hashlib.sha1()
    joined = "\n---\n".join(parts)
    h.update(joined.encode("utf-8"))
    return h.hexdigest()


def extract_primary_command(section_title: str, block_title: Optional[str]) -> Optional[str]:
    """
    Try to infer the primary command string for metadata.
    Priority:
      1) bold block title (often the command itself)
      2) section title patterns like "MACsec show command" -> "show macsec" (best-effort)
    """
    if block_title:
        return block_title.strip()

    # Best-effort: map "... show command" -> "show <feature>"
    m = re.search(r"^(?P<feature>.+?)\s+show command$", section_title.strip(), re.IGNORECASE)
    if m:
        feature = m.group("feature").strip()
        # Common SONiC pattern: feature name first, command verb fixed
        # e.g., "MACsec show command" -> "show macsec"
        return f"show {feature.lower()}"

    m = re.search(r"^(?P<feature>.+?)\s+config command$", section_title.strip(), re.IGNORECASE)
    if m:
        feature = m.group("feature").strip()
        return f"config {feature.lower()}"

    m = re.search(r"^(?P<feature>.+?)\s+clear command$", section_title.strip(), re.IGNORECASE)
    if m:
        feature = m.group("feature").strip()
        # SONiC uses sonic-clear for many features
        return f"sonic-clear {feature.lower()}"

    return None


# -----------------------------
# Core splitting logic
# -----------------------------
def split_into_h2_sections(md: str) -> List[Dict]:
    """
    Return list of dicts: {section_title, section_text}
    """
    md = md.replace("\r\n", "\n").replace("\r", "\n")

    matches = list(H2_RE.finditer(md))
    if not matches:
        # If no H2 headings, treat whole doc as one section
        return [{"section_title": "Document", "section_text": md}]

    sections = []
    for idx, m in enumerate(matches):
        start = m.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(md)
        section_title = m.group("title").strip()
        section_text = md[start:end].strip()
        sections.append({"section_title": section_title, "section_text": section_text})
    return sections


def split_section_by_bold_commands(section_title: str, section_text: str) -> List[Dict]:
    """
    Within an H2 section, split further by bold command markers (**...**),
    but keep small sections intact if no bold marker exists.

    Output blocks: {block_title, block_text}
    """
    # Remove the leading H2 line from block splitting so it doesn't confuse bold headings
    section_body = re.sub(rf"^##\s+{re.escape(section_title)}\s*$", "", section_text, count=1, flags=re.M).strip()

    bolds = list(BOLD_CMD_RE.finditer(section_body))
    if not bolds:
        # No bold command markers: keep as a single block
        return [{"block_title": None, "block_text": section_body.strip()}]

    blocks = []
    for idx, m in enumerate(bolds):
        block_title = m.group("cmd").strip()
        start = m.start()
        end = bolds[idx + 1].start() if idx + 1 < len(bolds) else len(section_body)
        block_text = section_body[start:end].strip()
        blocks.append({"block_title": block_title, "block_text": block_text})
    return blocks


def further_split_large_block(block_text: str, max_chars: int = 2500) -> List[str]:
    """
    If a block is very large, split conservatively at blank lines,
    but never break fenced code blocks.

    This keeps CLI examples intact.
    """
    text = block_text.strip()
    if len(text) <= max_chars:
        return [text]

    lines = text.split("\n")
    parts = []
    cur = []
    in_fence = False

    def flush():
        nonlocal cur
        if cur:
            parts.append("\n".join(cur).strip())
            cur = []

    for line in lines:
        # fence toggle
        if line.strip().startswith("```"):
            in_fence = not in_fence
            cur.append(line)
            continue

        # split point: blank line and not inside code block and current is big enough
        if (not in_fence) and (line.strip() == "") and (sum(len(x) + 1 for x in cur) >= max_chars):
            flush()
            continue

        cur.append(line)

    flush()

    # If still too large (rare), fall back to hard split without breaking fences
    final_parts = []
    for p in parts:
        if len(p) <= max_chars:
            final_parts.append(p)
        else:
            # hard split, but keep fences intact by splitting on lines
            plines = p.split("\n")
            acc = []
            acc_len = 0
            in_fence2 = False
            for ln in plines:
                if ln.strip().startswith("```"):
                    in_fence2 = not in_fence2
                acc.append(ln)
                acc_len += len(ln) + 1
                if (not in_fence2) and acc_len >= max_chars:
                    final_parts.append("\n".join(acc).strip())
                    acc, acc_len = [], 0
            if acc:
                final_parts.append("\n".join(acc).strip())
    return [x for x in final_parts if x.strip()]


def build_chunks(md_path: str, out_jsonl: str) -> None:
    md = Path(md_path).read_text(encoding="utf-8")

    h2_sections = split_into_h2_sections(md)
    out = Path(out_jsonl)
    out.parent.mkdir(parents=True, exist_ok=True)

    chunks: List[Chunk] = []

    for sec in h2_sections:
        section_title = sec["section_title"]
        section_text = sec["section_text"]

        blocks = split_section_by_bold_commands(section_title, section_text)

        for b in blocks:
            block_title = b["block_title"]
            block_text = b["block_text"].strip()
            if not block_text:
                continue

            # If big, split but preserve code blocks
            sub_texts = further_split_large_block(block_text, max_chars=2500)

            for sub_idx, sub_text in enumerate(sub_texts, start=1):
                # Create human-friendly title
                title = block_title or section_title
                if len(sub_texts) > 1:
                    title = f"{title} (part {sub_idx})"

                command = extract_primary_command(section_title, block_title)

                # Build content with light semantic wrapper
                content = (
                    f"Section: {section_title}\n"
                    + (f"Command: {command}\n" if command else "")
                    + "\n"
                    + sub_text.strip()
                )

                norm = normalize_for_id(content)
                cid = sha1_id("Command-Reference.md", section_title, title, norm)

                meta = {
                    "source": "Command-Reference.md",
                    "section": section_title,
                    "title": title,
                    "type": "cli_reference",
                }
                if command:
                    meta["command"] = command
                    meta["verb"] = command.split(maxsplit=1)[0]

                chunks.append(
                    Chunk(
                        id=cid,
                        title=title,
                        section=section_title,
                        command=command,
                        content=content,
                        meta=meta,
                    )
                )

    # Write JSONL
    with out.open("w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(asdict(c), ensure_ascii=False) + "\n")

    print(f"Wrote {len(chunks)} chunks to {out_jsonl}")


if __name__ == "__main__":
    # Change these paths as needed
    build_chunks(
        md_path="/data/chengxi/Haystack/Command-Reference.md",
        out_jsonl="/data/chengxi/Haystack/chunks.jsonl",
    )


Wrote 526 chunks to /data/chengxi/Haystack/chunks.jsonl


In [5]:
import re
import json
import hashlib
from typing import Optional, Dict, Tuple
from haystack import Document

USAGE_RE = re.compile(r"(?im)^\s*Usage:\s*$")
OPTIONS_RE = re.compile(r"(?im)^\s*Options:\s*$")
EXAMPLE_RE = re.compile(r"(?im)^\s*(Example|Examples|Example Output|Sample Output):\s*$")
CODE_FENCE_RE = re.compile(r"(?m)^```")

def _normalize_ws(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

def _sha1_id(*parts: str) -> str:
    h = hashlib.sha1()
    h.update("\n---\n".join(parts).encode("utf-8"))
    return h.hexdigest()

def _best_effort_command(section: str, title: Optional[str]) -> Optional[str]:
    # Prefer explicit title if it looks like a command
    if title:
        t = title.strip()
        # Heuristic: command often starts with show/config/sonic-clear/vtysh
        if re.match(r"^(show|config|sonic-clear|vtysh)\b", t, re.IGNORECASE):
            return t

    # Fallback: infer from section title like "MACsec show command"
    m = re.search(r"^(?P<feature>.+?)\s+show command$", section.strip(), re.IGNORECASE)
    if m:
        return f"show {m.group('feature').strip().lower()}"
    m = re.search(r"^(?P<feature>.+?)\s+clear command$", section.strip(), re.IGNORECASE)
    if m:
        return f"sonic-clear {m.group('feature').strip().lower()}"
    m = re.search(r"^(?P<feature>.+?)\s+config command$", section.strip(), re.IGNORECASE)
    if m:
        return f"config {m.group('feature').strip().lower()}"
    return None

def _extract_subsections(raw: str) -> Dict[str, str]:
    """
    Best-effort extraction: try to split into description/usage/options/example/other.
    This is intentionally conservative; if markers not found, raw goes into 'body'.
    """
    raw = _normalize_ws(raw)

    # If no markers exist, return as body
    if not (USAGE_RE.search(raw) or OPTIONS_RE.search(raw) or EXAMPLE_RE.search(raw)):
        return {"body": raw}

    # Split by markers while preserving code fences
    lines = raw.split("\n")
    parts = {"description": [], "usage": [], "options": [], "example": [], "notes": []}
    cur = "description"
    in_fence = False

    def set_section(name: str):
        nonlocal cur
        cur = name

    for line in lines:
        if line.strip().startswith("```"):
            in_fence = not in_fence
            parts[cur].append(line)
            continue

        if not in_fence:
            if re.match(r"(?i)^\s*Usage:\s*$", line):
                set_section("usage")
                continue
            if re.match(r"(?i)^\s*Options:\s*$", line):
                set_section("options")
                continue
            if re.match(r"(?i)^\s*(Example|Examples|Example Output|Sample Output):\s*$", line):
                set_section("example")
                continue

        parts[cur].append(line)

    # Clean empties
    out = {}
    for k, v in parts.items():
        txt = _normalize_ws("\n".join(v))
        if txt:
            out[k] = txt
    return out

def build_haystack_document(
    *,
    section_title: str,
    chunk_title: Optional[str],
    raw_text: str,
    source: str = "Command-Reference.md",
    extra_meta: Optional[Dict] = None,
) -> Document:
    """
    Convert a raw CLI chunk into a high-quality Haystack Document.
    """
    command = _best_effort_command(section_title, chunk_title)
    verb = command.split(maxsplit=1)[0] if command else None

    subs = _extract_subsections(raw_text)

    # Intent line (helps retrieval + LLM grounding)
    intent = None
    st_lower = section_title.lower()
    if "show command" in st_lower:
        intent = "Display operational status / counters / configuration for this feature."
    elif "clear command" in st_lower:
        intent = "Clear counters / reset runtime statistics for this feature."
    elif "config command" in st_lower:
        intent = "Configure this feature (persistent configuration)."

    # Compose structured content (what you asked for)
    header_lines = [
        f"Section: {section_title}",
        f"Title: {chunk_title or section_title}",
    ]
    if command:
        header_lines.append(f"Command: {command}")
    if intent:
        header_lines.append(f"Intent: {intent}")

    body_blocks = []
    if "description" in subs:
        body_blocks.append("Description:\n" + subs["description"])
    if "usage" in subs:
        body_blocks.append("Usage:\n" + subs["usage"])
    if "options" in subs:
        body_blocks.append("Options:\n" + subs["options"])
    if "example" in subs:
        body_blocks.append("Example / Output:\n" + subs["example"])

    # Anything else not classified goes to Body
    if "body" in subs and subs["body"]:
        body_blocks.append("Body:\n" + subs["body"])
    if "notes" in subs:
        body_blocks.append("Notes:\n" + subs["notes"])

    content = _normalize_ws("\n".join(header_lines) + "\n\n" + "\n\n".join(body_blocks))

    meta = {
        "source": source,
        "domain": "SONiC-CLI",
        "section": section_title,
        "title": chunk_title or section_title,
        "type": "cli_reference",
    }
    if command:
        meta["command"] = command
    if verb:
        meta["verb"] = verb

    if extra_meta:
        meta.update(extra_meta)

    # Stable ID: good for insert-only semantics + dedup
    doc_id = _sha1_id(source, section_title, chunk_title or "", content)

    return Document(id=doc_id, content=content, meta=meta)


In [7]:
import json
from pathlib import Path
from haystack import Document


def build_haystack_document(
    *,
    section_title: str,
    chunk_title: Optional[str],
    raw_text: str,
    source: str = "Command-Reference.md",
    extra_meta: Optional[Dict] = None,
) -> Document:
    """
    Convert a raw CLI chunk into a high-quality Haystack Document.
    """
    command = _best_effort_command(section_title, chunk_title)
    verb = command.split(maxsplit=1)[0] if command else None

    subs = _extract_subsections(raw_text)

    # Intent line (helps retrieval + LLM grounding)
    intent = None
    st_lower = section_title.lower()
    if "show command" in st_lower:
        intent = "Display operational status / counters / configuration for this feature."
    elif "clear command" in st_lower:
        intent = "Clear counters / reset runtime statistics for this feature."
    elif "config command" in st_lower:
        intent = "Configure this feature (persistent configuration)."

    # Compose structured content (what you asked for)
    header_lines = [
        f"Section: {section_title}",
        f"Title: {chunk_title or section_title}",
    ]
    if command:
        header_lines.append(f"Command: {command}")
    if intent:
        header_lines.append(f"Intent: {intent}")

    body_blocks = []
    if "description" in subs:
        body_blocks.append("Description:\n" + subs["description"])
    if "usage" in subs:
        body_blocks.append("Usage:\n" + subs["usage"])
    if "options" in subs:
        body_blocks.append("Options:\n" + subs["options"])
    if "example" in subs:
        body_blocks.append("Example / Output:\n" + subs["example"])

    # Anything else not classified goes to Body
    if "body" in subs and subs["body"]:
        body_blocks.append("Body:\n" + subs["body"])
    if "notes" in subs:
        body_blocks.append("Notes:\n" + subs["notes"])

    content = _normalize_ws("\n".join(header_lines) + "\n\n" + "\n\n".join(body_blocks))

    meta = {
        "source": source,
        "domain": "SONiC-CLI",
        "section": section_title,
        "title": chunk_title or section_title,
        "type": "cli_reference",
    }
    if command:
        meta["command"] = command
    if verb:
        meta["verb"] = verb

    if extra_meta:
        meta.update(extra_meta)

    # Stable ID: good for insert-only semantics + dedup
    doc_id = _sha1_id(source, section_title, chunk_title or "", content)

    return Document(id=doc_id, content=content, meta=meta)
  # 若你把函数放在单独文件里

chunks_path = Path("/data/chengxi/Haystack/chunks.jsonl")
docs = []

for line in chunks_path.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    obj = json.loads(line)
    section = obj.get("section") or obj.get("meta", {}).get("section") or "Unknown"
    title = obj.get("title") or obj.get("meta", {}).get("title")
    raw = obj.get("content", "")

    doc = build_haystack_document(
        section_title=section,
        chunk_title=title,
        raw_text=raw,
        source="Command-Reference.md",
        extra_meta=obj.get("meta", {}),
    )
    docs.append(doc)

print("Documents:", len(docs))
print(docs[0].content[:500])


Documents: 526
Section: Table of Contents
Title: Table of Contents (part 1)

Body:
Section: Table of Contents

* [Document History](#document-history)
* [Introduction](#introduction)
* [Basic Tasks](#basic-tasks)
  * [SSH Login](#ssh-login)
  * [Show Management Interface](#show-management-interface)
  * [Configuring Management Interface](#configuring-management-interface)
* [Getting Help](#getting-help)
  * [Help for Config Commands](#help-for-config-commands)
  * [Help for Show Commands](#help-for-show-comman


In [9]:
from haystack import Document
import hashlib
import re


def build_cli_document(
    *,
    chunk_text: str,
    section: str,
    command: str | None = None,
    source: str = "Command-Reference.md",
):
    """
    Build a high-quality Haystack Document for SONiC CLI reference.
    """

    # ---------- 1. 轻量语义包装（关键） ----------
    header_lines = [
        f"Section: {section}",
    ]
    if command:
        header_lines.append(f"Command: {command}")

    content = (
        "\n".join(header_lines)
        + "\n\n"
        + chunk_text.strip()
    )

    # ---------- 2. 稳定 ID（insert-only 友好） ----------
    # 同一文档同一 chunk => 同一 id
    doc_id = hashlib.sha1(
        (source + section + (command or "") + content).encode("utf-8")
    ).hexdigest()

    # ---------- 3. metadata（用于检索约束 & 防幻觉） ----------
    meta = {
        "source": source,
        "domain": "SONiC-CLI",
        "section": section,
        "type": "cli_reference",
    }

    if command:
        meta["command"] = command
        meta["verb"] = command.split(maxsplit=1)[0]  # show / config / sonic-clear

    return Document(
        id=doc_id,
        content=content,
        meta=meta,
    )



In [10]:
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.document_stores.types import DuplicatePolicy

store = InMemoryDocumentStore()

store.write_documents(
    docs,
    policy=DuplicatePolicy.SKIP  # 同内容不重复插入
)


526

In [18]:
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

# 用于 query 的 embedder（这是关键）
text_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
text_embedder.warm_up()

retriever = InMemoryEmbeddingRetriever(document_store=store, top_k=3)

query_emb = text_embedder.run(
    text="How to Disable Memory Statistics Monitoring?"
)["embedding"]

docs = retriever.run(query_embedding=query_emb)["documents"]

for d in docs:
    print(d.meta.get("command"), "->", d.content[:80])



Batches: 100%|██████████| 1/1 [00:00<00:00, 200.64it/s]

None -> Section: Memory Statistics Config Commands
Title: Memory Statistics Config Comma
Time Format for Statistics Retrieval: -> Section: Memory Statistics Show Commands
Title: Time Format for Statistics Retri
Command Options: -> Section: Memory Statistics Show Commands
Title: Command Options:
Intent: Display
